# Inference of the Tensor-to-Scalar Ratio

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CMBSciPol/jax-healpy/blob/main/docs/user-guide/inference.ipynb)

Primordial gravitational waves, if produced during inflation, imprint B modes in the
polarization of the CMB, with an amplitude set by the tensor-to-scalar ratio $r$.
Gravitational lensing by the large-scale structure also turns E modes into B modes, with an
amplitude $A_\mathrm{lens}$ relative to the $\Lambda$CDM prediction. This page infers
$r$ and $A_\mathrm{lens}$ from a simulated polarized sky, using the three tools that JAX
brings to the analysis:

- the gradient and the Fisher matrix of the likelihood, by automatic differentiation;
- Hamiltonian Monte Carlo sampling of the posterior, with [NumPyro](https://num.pyro.ai);
- the derivative of a whole simulation, from the power spectra to the maps and back,
  with respect to $r$.

!!! note
    This page requires the `recommended` extra (s2fft), see the
    [installation](installation.md) page, and NumPyro: `pip install numpyro`.

In [ ]:
# Install jax-healpy and its dependencies on Google Colab
import sys

if 'google.colab' in sys.modules:
    %pip install --quiet "jax-healpy[cpu] @ git+https://github.com/CMBSciPol/jax-healpy" healpy numpyro
    !wget --quiet --directory-prefix=data https://raw.githubusercontent.com/CMBSciPol/jax-healpy/main/docs/user-guide/data/cmb_cls.txt
    !wget --quiet --directory-prefix=data https://raw.githubusercontent.com/CMBSciPol/jax-healpy/main/docs/user-guide/data/bb_templates.txt

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS

import jax_healpy as hp

# Enable 64-bit precision (recommended, jax-healpy warns otherwise)
jax.config.update('jax_enable_x64', True)

## Simulated Observation

The spectra were computed with [CAMB](https://camb.info) for the Planck 2018 best-fit
$\Lambda$CDM parameters and $r = 0.01$ (see the [polarization](polarization.md) page).
The B-mode spectrum is the sum of two templates, from lensing and from gravitational waves
for $r = 1$:

$$
C_\ell^{BB}(r, A_\mathrm{lens}) = A_\mathrm{lens}\, C_\ell^\mathrm{lens} + r\, C_\ell^\mathrm{tensor}.
$$

We observe the full sky at `nside = 64`, with a white noise of 3 µK-arcmin in $Q$ and $U$,
the order of magnitude targeted by the next generation of CMB satellites, without beam or
foregrounds. Its power spectrum is $N_\ell = \sigma^2$, where $\sigma$ is the noise level in
µK·rad.

In [ ]:
nside = 64
lmax = 2 * nside
ell, cl_tt, cl_ee, cl_bb, cl_te = np.loadtxt('data/cmb_cls.txt')[: lmax + 1].T
_, bb_lens, bb_tensor = np.loadtxt('data/bb_templates.txt')[: lmax + 1].T

noise_level = np.radians(3 / 60)  # 3 µK-arcmin in µK·rad
noise_cl = noise_level**2 * np.ones(lmax + 1)
noise_std = noise_level / np.sqrt(hp.nside2pixarea(nside))  # per pixel, in µK

r_true, alens_true = 0.01, 1.0
key_cmb, key_noise = jax.random.split(jax.random.key(2026))
cmb_maps = hp.synfast(key_cmb, [cl_tt, cl_ee, cl_bb, cl_te], nside, lmax=lmax, new=True)
qu_maps = cmb_maps[1:] + noise_std * jax.random.normal(key_noise, (2, hp.nside2npix(nside)))
print(f'Noise per pixel: {noise_std:.2f} µK')

`anafast` computes the $EE$, $BB$ and $EB$ spectra of the $Q, U$ maps. We fit the
$BB$ spectrum from $\ell = 2$, where the reionization bump at $\ell < 10$ and the
recombination bump at $\ell \simeq 80$ carry the information on $r$.

In [ ]:
cl_ee_obs, cl_bb_obs, cl_eb_obs = hp.anafast(qu_maps, lmax=lmax)

dl = ell * (ell + 1) / (2 * np.pi)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(ell[2:], (dl * cl_bb_obs)[2:], '.', color='0.3', label='Observed')
ax.plot(ell[2:], (dl * (bb_lens + r_true * bb_tensor + noise_cl))[2:], color='k', label='Total')
ax.plot(ell[2:], (dl * bb_lens)[2:], label='Lensing')
ax.plot(ell[2:], (dl * r_true * bb_tensor)[2:], label=f'Tensor, r = {r_true}')
ax.plot(ell[2:], (dl * noise_cl)[2:], '--', label='Noise')
ax.set(xscale='log', yscale='log', xlabel=r'$\ell$', ylabel=r'$D_\ell^{BB}$ [µK$^2$]', ylim=(1e-6, 1e-1))
ax.legend()
plt.show()

## Likelihood

On the full sky, the $2\ell + 1$ coefficients $a^B_{\ell m}$ of each multipole are
independent Gaussian variables of variance $C_\ell = C_\ell^{BB} + N_\ell$. Up to a
constant, the negative log-likelihood of the observed spectrum $\hat C_\ell$ is

$$
-\ln \mathcal{L}(r, A_\mathrm{lens}) = \sum_{\ell \ge 2} \frac{2\ell + 1}{2}
\left[\frac{\hat C_\ell}{C_\ell} + \ln C_\ell\right].
$$

In [ ]:
ell_fit = ell[2:]
observed = cl_bb_obs[2:]


def model_cl(params):
    """The B-mode spectrum, with noise, for params = (r, A_lens)."""
    r, alens = params
    return alens * bb_lens[2:] + r * bb_tensor[2:] + noise_cl[2:]


def neg_log_likelihood(params):
    cl = model_cl(params)
    return jnp.sum((2 * ell_fit + 1) / 2 * (observed / cl + jnp.log(cl)))

## Maximum Likelihood

We maximize the likelihood by Fisher scoring, a Newton method in which the Hessian is
replaced by its expectation, the Fisher matrix:

$$
F_{ij} = \sum_{\ell \ge 2} \frac{2\ell + 1}{2}
\frac{1}{C_\ell^2} \frac{\partial C_\ell}{\partial \theta_i} \frac{\partial C_\ell}{\partial \theta_j}.
$$

JAX provides the derivatives of the model with `jax.jacobian` and the gradient of the
likelihood with `jax.grad`: neither is written by hand. The inverse of the Fisher matrix at
the maximum estimates the covariance of the parameters.

In [ ]:
@jax.jit
def fisher_matrix(params):
    derivatives = jax.jacobian(model_cl)(params)  # shape (n_ell, 2)
    weights = (2 * ell_fit + 1) / 2 / model_cl(params) ** 2
    return jnp.einsum('l,li,lj->ij', weights, derivatives, derivatives)


@jax.jit
def fisher_scoring_step(params):
    return params - jnp.linalg.solve(fisher_matrix(params), jax.grad(neg_log_likelihood)(params))


params = jnp.array([0.0, 1.0])
for iteration in range(5):
    params = fisher_scoring_step(params)
    print(f'Iteration {iteration}: r = {params[0]:.5f}, A_lens = {params[1]:.4f}')

covariance = jnp.linalg.inv(fisher_matrix(params))
r_ml, alens_ml = params
sigma_r, sigma_alens = jnp.sqrt(jnp.diag(covariance))
correlation = covariance[0, 1] / (sigma_r * sigma_alens)
print(f'r = {r_ml:.4f} ± {sigma_r:.4f}, A_lens = {alens_ml:.3f} ± {sigma_alens:.3f}, correlation {correlation:.2f}')

The iterations converge in a few steps. The parameters agree with the true values,
$r = 0.01$ and $A_\mathrm{lens} = 1$, within their uncertainties. They are
anti-correlated: both contributions add power to the B modes.

## Posterior Sampling with NUTS

The Fisher matrix describes the posterior only near its maximum. To sample the full
posterior, we use the No-U-Turn Sampler (NUTS) of NumPyro, a Hamiltonian Monte Carlo method
that follows the gradient of the log-posterior, provided by JAX. The likelihood enters the
NumPyro model as a factor, with uniform priors $0 \le r \le 0.1$ and
$0 \le A_\mathrm{lens} \le 2$.

In [ ]:
def model():
    r = numpyro.sample('r', dist.Uniform(0, 0.1))
    alens = numpyro.sample('A_lens', dist.Uniform(0, 2))
    numpyro.factor('likelihood', -neg_log_likelihood(jnp.stack([r, alens])))


mcmc = MCMC(NUTS(model), num_warmup=500, num_samples=4000, progress_bar=False)
mcmc.run(jax.random.key(0))
samples = mcmc.get_samples()
for name in ['r', 'A_lens']:
    print(f'{name} = {samples[name].mean():.4f} ± {samples[name].std():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, name, mean, sigma in zip(axes, ['r', 'A_lens'], [r_ml, alens_ml], [sigma_r, sigma_alens]):
    ax.hist(samples[name], bins=50, density=True, color='0.75', label='NUTS')
    x = np.linspace(mean - 4 * sigma, mean + 4 * sigma, 200)
    ax.plot(x, np.exp(-0.5 * ((x - mean) / sigma) ** 2) / (np.sqrt(2 * np.pi) * sigma), 'k', label='Fisher')
    ax.axvline({'r': r_true, 'A_lens': alens_true}[name], color='C3', label='True value')
    ax.set(xlabel=name.replace('A_lens', r'$A_\mathrm{lens}$'), yticks=[])
axes[0].legend()
axes[2].plot(samples['r'], samples['A_lens'], ',', color='0.4', alpha=0.5)
axes[2].plot(r_true, alens_true, '+', color='C3', ms=14, mew=2)
axes[2].set(xlabel='r', ylabel=r'$A_\mathrm{lens}$')
plt.tight_layout()
plt.show()

With this sensitivity, the posterior is close to the Gaussian approximation of the
Fisher matrix. NUTS remains exact when the posterior is not Gaussian, for example at a
lower $r$, where the positivity prior truncates the posterior, or with more parameters, such
as foreground amplitudes. Its cost grows slowly with the number of parameters, thanks to
the gradients.

## Differentiating a Simulation

The likelihood above is analytic. When no likelihood is known, for example on a masked sky
or with foregrounds, the analysis relies on simulations. With jax-healpy, a simulation is a
differentiable function of its parameters: for a fixed random key, the spectrum of a
simulated map can be differentiated with respect to $r$, through `synfast`, the spherical
harmonic transforms and `anafast`.

`jax.jvp` computes the derivative of the $129$ multipoles of the B-mode spectrum in a single
pass, at about the cost of a simulation. We check it against finite differences.

In [ ]:
@jax.jit
def simulated_cl_bb(r):
    """The B-mode spectrum of a CMB simulation, for a fixed random key."""
    cl_bb_model = alens_true * bb_lens + r * bb_tensor
    maps = hp.synfast(key_cmb, [cl_tt, cl_ee, cl_bb_model, cl_te], nside, lmax=lmax, new=True)
    return hp.anafast(maps[1:], lmax=lmax)[1]


cl_bb_sim, derivative = jax.jvp(simulated_cl_bb, (r_true,), (1.0,))

step = 1e-4
finite_difference = (simulated_cl_bb(r_true + step) - simulated_cl_bb(r_true - step)) / (2 * step)
error = jnp.max(jnp.abs(derivative - finite_difference)) / jnp.max(jnp.abs(finite_difference))
print(f'Relative difference with the finite differences: {error:.1e}')

Such derivatives make gradient-based methods possible where only simulations are
available: simulation-based inference, field-level inference, or the optimization of the
analysis itself, such as a mask or a filter.